# Q5 · Thương vong và thiếu thốn kinh tế – xã hội

**Phụ trách** C · **Tiêu chí** D1, D2, D3, D4, D5

Trong giai đoạn 2011–2015, số người bị nạn trên mỗi vụ và tỷ lệ trẻ em/người đi bộ khác nhau thế nào giữa các nhóm IMD có dữ liệu?

[Câu hỏi và lộ trình chi tiết](../question.md)

## 1. Phạm vi và dữ liệu

Ghi người dùng kết quả, đơn vị quan sát và phạm vi.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Notebook có thể chạy từ thư mục gốc hoặc từ thư mục notebooks/.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
if not (project_root / "environment.yml").is_file():
    raise FileNotFoundError("Mở Jupyter Notebook từ thư mục gốc của dự án.")

raw_dir = project_root / "data" / "raw"
processed_dir = project_root / "data" / "processed"

# Cho phép import src/ khi kernel chạy trong thư mục notebooks/.
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
from src.data_prep.road_safety import preview_table

inputs = {
    "accident_counts.csv": ['accident_id', 'accident_year', 'n_casualties', 'urban_or_rural_area', 'accident_imd_decile'],
    "vehicle_accident.csv": ['accident_id', 'vehicle_ref', 'accident_year', 'driver_imd_decile'],
    "casualty_vehicle.csv": ['accident_id', 'vehicle_ref', 'casualty_ref', 'accident_year', 'age', 'casualty_type', 'casualty_imd_decile'],
}

# Chỉ đọc mẫu để kiểm tra schema. Dùng toàn bộ mẫu hợp lệ khi phân tích.
for filename, required_columns in inputs.items():
    data_path = processed_dir / filename
    if not data_path.is_file():
        print(f"Chưa có {filename}. Chạy notebook 01 để tạo bảng.")
        continue
    preview_table(data_path, required_columns)
    print(f"Đã kiểm tra cột đầu vào của {filename}.")

## 2. Chọn mẫu và kiểm tra dữ liệu

Kiểm tra độ phủ IMD theo năm và theo nguồn. Phân biệt IMD tại nơi xảy ra vụ với IMD nơi cư trú của tài xế hoặc người bị nạn. Nếu tệp không có IMD phù hợp thì ghi hạn chế, xác minh nguồn bổ sung và dừng phần ước lượng tương ứng. Ghi số dòng trước và sau mỗi bước.

In [ ]:
imd_inputs = {
    "accident_counts.csv": "accident_imd_decile",
    "vehicle_accident.csv": "driver_imd_decile",
    "casualty_vehicle.csv": "casualty_imd_decile",
}

# Đếm IMD hợp lệ trong toàn giai đoạn nghiên cứu, không suy từ 5 dòng mẫu.
for filename, field in imd_inputs.items():
    path = processed_dir / filename
    if not path.is_file():
        continue
    total = valid = 0
    chunks = pd.read_csv(path, usecols=["accident_year", field], chunksize=200_000)
    for chunk in chunks:
        in_scope = chunk.loc[chunk["accident_year"].between(2011, 2015), field]
        total += len(in_scope)
        valid += int(in_scope.between(1, 10).sum())
    print(f"{field}: {valid:,}/{total:,} dòng có IMD hợp lệ trong 2011–2015.")
    if not valid:
        print("Cần xác minh thêm nguồn IMD trước khi phân tích trường này.")

## 3. Phương pháp

Với dữ liệu đủ phủ, phân nhóm 1–3, 4–7 và 8–10. Tính riêng chỉ số cấp vụ và tỷ lệ trẻ em/người đi bộ ở cấp người. So sánh xu hướng 2011–2015, phân tầng đô thị/nông thôn và báo kích thước tác động cùng kiểm định phù hợp.

In [ ]:
# Thực hiện các phép tính và kiểm tra bảng trung gian.

## 4. Kết quả và biểu đồ

Vẽ độ phủ và xu hướng/chênh lệch có mẫu số. Ghi phạm vi địa lý và mức độ so sánh được của các chỉ số IMD.

In [ ]:
# Thêm kết quả thực nghiệm và biểu đồ.

## 5. Trả lời và giới hạn

Tự viết nhận xét dựa trên kết quả, mẫu số, độ phủ và độ bất định.